# Notebook 66 — ML Baselines vs Sparse Symbolic Chart

Notebook 65 showed that metadata structure is real: shuffling regime labels degrades performance.

Notebook 66 answers the next reviewer-style question:

> Is the sparse symbolic chart still competitive against flexible ML baselines under hard regime shifts?

We compare:

1. `pruned_symbolic`
2. `ridge_chart`
3. `random_forest`
4. `gradient_boosting`
5. `small_mlp`

The target is the coefficient vector:

\[
\beta =
(\beta_0,\beta_c,\beta_r,\beta_{c^3},\beta_{r^2},\beta_{rc^2})
\]

for the shared governing field:

\[
g(r,c;\beta)
=
\beta_0+\beta_c c+\beta_r r+\beta_{c^3}c^3+\beta_{r^2}r^2+\beta_{rc^2}rc^2.
\]

## Main test

Flexible ML may fit locally, but the paper-level question is hard-block generalization:

\[
\text{sparse symbolic chart} \stackrel{?}{\geq}
\text{ML baselines under regime shift}.
\]

## Outputs

- Hard-block RMSE comparisons
- Universality score
- Win-rate table
- Train vs holdout generalization gap
- Shuffle ablation for top models
- Paper-ready conclusion paragraph

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LassoCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, LeaveOneOut
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

np.random.seed(42)

OUTPUT_DIR = "paper66_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
})

## 1. Load data or synthetic fallback

This notebook is standalone. It rebuilds the same coefficient table as Notebooks 64–65.

In [ ]:
DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append({
                                    "system": system,
                                    "task": task,
                                    "forcing_mode": forcing_mode,
                                    "k": k,
                                    "flow_mode": flow_mode,
                                    "condition_coord": c,
                                    "residual": r,
                                    "predicted_flow": g + noise,
                                    "next_residual": next_r,
                                    "delta_condition": delta_c,
                                    "sample_id": sample_id,
                                    "path_id": path_id,
                                    "window_id": window_id,
                                })
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        df["delta_condition"] = np.gradient(df["condition_coord"].to_numpy())

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for key, value in defaults.items():
        if key not in df.columns:
            df[key] = value

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)

print("Data source:", data_source)
print("Synthetic fallback:", USE_SYNTHETIC_FALLBACK)
print("Rows:", len(df))
display(df.head())

## 2. Shared governing field and coefficient table

In [ ]:
TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def governing_field(r, c, beta):
    beta = np.asarray(beta, dtype=float)
    return (
        beta[0]
        + beta[1] * c
        + beta[2] * r
        + beta[3] * c**3
        + beta[4] * r**2
        + beta[5] * r * c**2
    )

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "template_rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ np.asarray(beta, dtype=float)

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i + 1] - cgrid[i])
            g = float(np.clip(governing_field(r, c, beta), -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    return float(np.mean([
        math.sqrt(mean_squared_error(integrate(beta_ref, r0), integrate(beta_cmp, r0)))
        for r0 in r0s
    ]))

group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue

    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"

    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)

    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
coef_df.to_csv(os.path.join(OUTPUT_DIR, "coefficient_table.csv"), index=False)

print("Regime count:", len(coef_df))
display(coef_df.head())

## 3. Feature construction and symbolic pruning

In [ ]:
def build_symbolic_features(df_in, columns=None, reduced_terms=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)

    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2

    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    X_ff = pd.get_dummies(ff, prefix="ff")

    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X_sf = pd.get_dummies(sf, prefix="sf")

    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(
        df_in["k"].astype(float).to_numpy().reshape(-1, 1)
    )

    X = pd.concat([X, X_ff, X_sf, X_fk], axis=1)

    if reduced_terms is not None:
        X = X.reindex(columns=reduced_terms, fill_value=0.0)

    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)

    return X.astype(float)

def term_stability_table(df_in, n_splits=8, threshold=1e-4):
    n = len(df_in)
    splitter = LeaveOneOut() if n <= 12 else KFold(n_splits=min(n_splits, n), shuffle=True, random_state=42)

    all_features = build_symbolic_features(df_in).columns.tolist()
    stability = {coef: {feat: 0 for feat in all_features} for coef in COEF_COLS}
    fold_count = 0

    for train_idx, _ in splitter.split(df_in):
        train_df = df_in.iloc[train_idx].reset_index(drop=True)
        X_train = build_symbolic_features(train_df, columns=all_features)

        for coef in COEF_COLS:
            y = train_df[coef].to_numpy(dtype=float)
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X_train)
            model = LassoCV(cv=min(5, len(train_df)), random_state=fold_count + 1, max_iter=30000)
            model.fit(Xs, y)

            for feat, val in zip(all_features, model.coef_):
                if abs(val) > threshold:
                    stability[coef][feat] += 1

        fold_count += 1

    rows = []
    for coef in COEF_COLS:
        for feat in all_features:
            rows.append({
                "coefficient": coef,
                "term": feat,
                "frequency": stability[coef][feat] / fold_count,
                "count": stability[coef][feat],
                "folds": fold_count,
            })
    return pd.DataFrame(rows)

stability_df = term_stability_table(coef_df)
STABILITY_THRESHOLD = 0.5

stable_terms_by_coef = {}
for coef in COEF_COLS:
    sub = stability_df[
        (stability_df["coefficient"] == coef)
        & (stability_df["frequency"] >= STABILITY_THRESHOLD)
    ]
    stable_terms_by_coef[coef] = sub["term"].tolist()

stable_terms_table = pd.DataFrame([
    {"coefficient": coef, "n_terms": len(stable_terms_by_coef[coef]), "terms": ", ".join(stable_terms_by_coef[coef])}
    for coef in COEF_COLS
])

display(stable_terms_table)
stable_terms_table.to_csv(os.path.join(OUTPUT_DIR, "stable_terms_by_coefficient.csv"), index=False)

## 4. Prediction model definitions

All models receive the same metadata feature input.

The symbolic model is the pruned chart. Flexible ML models are multi-output coefficient predictors.

In [ ]:
def predict_pruned_symbolic(train_df, test_df, terms_by_coef):
    Yhat = np.zeros((len(test_df), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        terms = terms_by_coef.get(coef, [])

        if len(terms) == 0:
            Yhat[:, j] = train_df[coef].mean()
            continue

        X_train = build_symbolic_features(train_df, reduced_terms=terms)
        X_test = build_symbolic_features(test_df, columns=X_train.columns, reduced_terms=terms)

        y_train = train_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)

        model = Ridge(alpha=1.0)
        model.fit(Xtr, y_train)
        Yhat[:, j] = model.predict(Xte)

    return Yhat

def predict_ridge_chart(train_df, test_df):
    X_train = build_symbolic_features(train_df)
    X_test = build_symbolic_features(test_df, columns=X_train.columns)
    model = Ridge(alpha=1.0)
    model.fit(X_train, train_df[COEF_COLS].to_numpy(dtype=float))
    return model.predict(X_test)

def predict_random_forest(train_df, test_df):
    X_train = build_symbolic_features(train_df)
    X_test = build_symbolic_features(test_df, columns=X_train.columns)
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=2,
        random_state=42,
    )
    model.fit(X_train, train_df[COEF_COLS].to_numpy(dtype=float))
    return model.predict(X_test)

def predict_gradient_boosting(train_df, test_df):
    X_train = build_symbolic_features(train_df)
    X_test = build_symbolic_features(test_df, columns=X_train.columns)
    base = GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2,
        random_state=42,
    )
    model = MultiOutputRegressor(base)
    model.fit(X_train, train_df[COEF_COLS].to_numpy(dtype=float))
    return model.predict(X_test)

def predict_small_mlp(train_df, test_df):
    X_train = build_symbolic_features(train_df)
    X_test = build_symbolic_features(test_df, columns=X_train.columns)

    model = Pipeline([
        ("scale", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(16,),
            activation="relu",
            alpha=1e-2,
            learning_rate_init=1e-3,
            max_iter=5000,
            random_state=42,
            early_stopping=False,
        )),
    ])
    model.fit(X_train, train_df[COEF_COLS].to_numpy(dtype=float))
    return model.predict(X_test)

MODEL_FUNCS = {
    "pruned_symbolic": lambda tr, te: predict_pruned_symbolic(tr, te, stable_terms_by_coef),
    "ridge_chart": predict_ridge_chart,
    "random_forest": predict_random_forest,
    "gradient_boosting": predict_gradient_boosting,
    "small_mlp": predict_small_mlp,
}

In [ ]:
def evaluate_predictions(test_df, predictions, block_name, split_name="holdout"):
    rows = []

    for i, rid in enumerate(test_df["regime_id"].tolist()):
        beta_true = test_df.loc[i, COEF_COLS].to_numpy(dtype=float)
        sub = regime_subsets[rid]
        y_emp = sub["predicted_flow"].to_numpy(dtype=float)

        for model_name, Yhat in predictions.items():
            beta_hat = Yhat[i]
            pred_field = predict_with_beta(sub, beta_hat)
            rows.append({
                "split": split_name,
                "block": block_name,
                "regime_id": rid,
                "model": model_name,
                "coef_rmse": math.sqrt(mean_squared_error(beta_true, beta_hat)),
                "field_rmse": math.sqrt(mean_squared_error(y_emp, pred_field)),
                "traj_rmse": trajectory_gap(sub, beta_true, beta_hat),
            })

    return rows

## 5. Hard-block evaluation

Use the same blocks as Notebook 65.

In [ ]:
hard_block_masks = {
    "k_high": coef_df["k"].astype(float) == 7.0,
    "k_low": coef_df["k"].astype(float) == 3.0,
    "forcing::condition": coef_df["forcing_mode"].astype(str) == "condition_gap",
    "forcing::capacity": coef_df["forcing_mode"].astype(str) == "capacity_gap",
    "forcing::feature": coef_df["forcing_mode"].astype(str) == "feature_gap",
    "system::entropy": coef_df["system"].astype(str) == "entropy",
    "system::unevenness": coef_df["system"].astype(str) == "unevenness",
    "flow::linear": coef_df["flow_mode"].astype(str) == "linear",
    "flow::nonlinear": coef_df["flow_mode"].astype(str) == "nonlinear",
}

holdout_rows = []

for block_name, test_mask in hard_block_masks.items():
    train_df = coef_df.loc[~test_mask].reset_index(drop=True)
    test_df = coef_df.loc[test_mask].reset_index(drop=True)

    if len(test_df) == 0 or len(train_df) < 5:
        continue

    predictions = {}
    for model_name, fn in MODEL_FUNCS.items():
        try:
            predictions[model_name] = fn(train_df, test_df)
        except Exception as e:
            print(f"Model {model_name} failed on {block_name}: {e}")

    holdout_rows.extend(evaluate_predictions(test_df, predictions, block_name, split_name="holdout"))

holdout_eval_df = pd.DataFrame(holdout_rows)
holdout_summary_df = holdout_eval_df.groupby(["block", "model"])[
    ["coef_rmse", "field_rmse", "traj_rmse"]
].mean().reset_index()

display(holdout_summary_df.sort_values(["block", "traj_rmse"]))
holdout_summary_df.to_csv(os.path.join(OUTPUT_DIR, "holdout_summary_by_block.csv"), index=False)

## 6. Figure 1 — Hard-block trajectory RMSE

In [ ]:
pivot = holdout_summary_df.pivot(index="block", columns="model", values="traj_rmse")

fig, ax = plt.subplots(figsize=(13, 5))
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("trajectory RMSE")
ax.set_title("Hard-block trajectory RMSE: symbolic vs ML baselines")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure1_hard_block_traj_rmse.png"), dpi=220, bbox_inches="tight")
plt.show()

# Also plot field RMSE
pivot_field = holdout_summary_df.pivot(index="block", columns="model", values="field_rmse")
fig, ax = plt.subplots(figsize=(13, 5))
pivot_field.plot(kind="bar", ax=ax)
ax.set_ylabel("field RMSE")
ax.set_title("Hard-block field RMSE: symbolic vs ML baselines")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure1b_hard_block_field_rmse.png"), dpi=220, bbox_inches="tight")
plt.show()

## 7. Universality score and win rates

In [ ]:
def universality_scores(summary_df, metric="traj_rmse", tolerance=0.05):
    rows = []
    for block, sub in summary_df.groupby("block"):
        best = sub[metric].min()
        for _, row in sub.iterrows():
            rows.append({
                "block": block,
                "model": row["model"],
                "metric": metric,
                "value": row[metric],
                "best": best,
                "within_tolerance": bool(row[metric] <= (1.0 + tolerance) * best),
                "is_best": bool(np.isclose(row[metric], best) or row[metric] == best),
            })

    block_df = pd.DataFrame(rows)
    score_df = block_df.groupby("model").agg(
        universality_score=("within_tolerance", "mean"),
        win_rate=("is_best", "mean"),
        mean_metric=("value", "mean"),
        n_blocks=("block", "nunique"),
    ).reset_index().sort_values(
        ["universality_score", "win_rate", "mean_metric"],
        ascending=[False, False, True],
    )

    return score_df, block_df

U_TOL = 0.05

universality_score_df, universality_block_df = universality_scores(
    holdout_summary_df,
    metric="traj_rmse",
    tolerance=U_TOL,
)

display(universality_score_df)
universality_score_df.to_csv(os.path.join(OUTPUT_DIR, "universality_score_ml_baselines.csv"), index=False)
universality_block_df.to_csv(os.path.join(OUTPUT_DIR, "universality_score_by_block.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(universality_score_df["model"], universality_score_df["universality_score"])
ax.set_ylim(0, 1.05)
ax.set_ylabel(f"Uτ, τ={U_TOL}")
ax.set_title("Universality score: symbolic vs ML baselines")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure2_universality_score_ml.png"), dpi=220, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(universality_score_df["model"], universality_score_df["win_rate"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("strict win rate")
ax.set_title("Block win rate: symbolic vs ML baselines")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure2b_win_rate_ml.png"), dpi=220, bbox_inches="tight")
plt.show()

## 8. Train vs holdout generalization gap

Flexible ML may fit training regimes but degrade under hard-block shifts. This section compares training and holdout coefficient RMSE.

In [ ]:
train_holdout_rows = []

for block_name, test_mask in hard_block_masks.items():
    train_df = coef_df.loc[~test_mask].reset_index(drop=True)
    test_df = coef_df.loc[test_mask].reset_index(drop=True)

    if len(test_df) == 0 or len(train_df) < 5:
        continue

    # Holdout predictions
    holdout_predictions = {}
    train_predictions = {}

    for model_name, fn in MODEL_FUNCS.items():
        try:
            holdout_predictions[model_name] = fn(train_df, test_df)
            train_predictions[model_name] = fn(train_df, train_df)
        except Exception as e:
            print(f"Model {model_name} failed on {block_name}: {e}")

    train_holdout_rows.extend(evaluate_predictions(train_df, train_predictions, block_name, split_name="train"))
    train_holdout_rows.extend(evaluate_predictions(test_df, holdout_predictions, block_name, split_name="holdout"))

train_holdout_df = pd.DataFrame(train_holdout_rows)
gap_summary = train_holdout_df.groupby(["model", "split"])[["coef_rmse", "field_rmse", "traj_rmse"]].mean().reset_index()
display(gap_summary)

gap_pivot = gap_summary.pivot(index="model", columns="split", values="coef_rmse")
gap_pivot["generalization_gap"] = gap_pivot["holdout"] - gap_pivot["train"]
gap_pivot = gap_pivot.sort_values("generalization_gap", ascending=False)
display(gap_pivot)

gap_summary.to_csv(os.path.join(OUTPUT_DIR, "train_holdout_summary.csv"), index=False)
gap_pivot.to_csv(os.path.join(OUTPUT_DIR, "generalization_gap_coef_rmse.csv"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(gap_pivot.index))
width = 0.35

ax.bar(x - width/2, gap_pivot["train"], width, label="train")
ax.bar(x + width/2, gap_pivot["holdout"], width, label="holdout")
ax.set_xticks(x)
ax.set_xticklabels(gap_pivot.index, rotation=25, ha="right")
ax.set_ylabel("coefficient RMSE")
ax.set_title("Train vs holdout coefficient RMSE")
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure3_train_vs_holdout_coef_rmse.png"), dpi=220, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(gap_pivot.index, gap_pivot["generalization_gap"])
ax.set_ylabel("holdout - train coefficient RMSE")
ax.set_title("Generalization gap")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure3b_generalization_gap.png"), dpi=220, bbox_inches="tight")
plt.show()

## 9. Shuffle ablation for top models

Run metadata shuffle on:

- `pruned_symbolic`
- best ML baseline by universality score
- second-best ML baseline if available

This tests whether flexible ML baselines depend on true metadata structure or exploit incidental fit capacity.

In [ ]:
def shuffled_metadata_copy(train_df, rng, cols_to_shuffle=("forcing_mode", "flow_mode", "system", "task", "k")):
    out = train_df.copy()
    for col in cols_to_shuffle:
        if col in out.columns:
            out[col] = rng.permutation(out[col].to_numpy())
    return out

# Select top ML baselines
ml_models = [m for m in universality_score_df["model"].tolist() if m not in ["pruned_symbolic", "ridge_chart"]]
selected_shuffle_models = ["pruned_symbolic"]
selected_shuffle_models.extend(ml_models[:2])
selected_shuffle_models = list(dict.fromkeys(selected_shuffle_models))

print("Selected models for shuffle ablation:", selected_shuffle_models)

In [ ]:
def run_shuffle_ablation(selected_models, n_repeats=20, seed=42):
    rng_master = np.random.default_rng(seed)
    rows = []

    for rep in range(n_repeats):
        rng = np.random.default_rng(int(rng_master.integers(0, 1_000_000_000)))

        for block_name, test_mask in hard_block_masks.items():
            train_df = coef_df.loc[~test_mask].reset_index(drop=True)
            test_df = coef_df.loc[test_mask].reset_index(drop=True)

            if len(test_df) == 0 or len(train_df) < 5:
                continue

            shuffled_train = shuffled_metadata_copy(train_df, rng)

            predictions = {}

            for model_name in selected_models:
                fn = MODEL_FUNCS[model_name]
                try:
                    predictions[f"{model_name}::true"] = fn(train_df, test_df)
                    predictions[f"{model_name}::shuffled"] = fn(shuffled_train, test_df)
                except Exception as e:
                    print(f"Shuffle model {model_name} failed on {block_name}: {e}")

            eval_rows = evaluate_predictions(test_df, predictions, block_name, split_name="shuffle_ablation")
            for row in eval_rows:
                row["repeat"] = rep
            rows.extend(eval_rows)

    return pd.DataFrame(rows)

shuffle_eval_df = run_shuffle_ablation(selected_shuffle_models, n_repeats=20, seed=42)

# parse model and condition
parts = shuffle_eval_df["model"].str.split("::", expand=True)
shuffle_eval_df["base_model"] = parts[0]
shuffle_eval_df["metadata_condition"] = parts[1]

shuffle_summary_df = shuffle_eval_df.groupby(["base_model", "metadata_condition"])[
    ["coef_rmse", "field_rmse", "traj_rmse"]
].mean().reset_index()

display(shuffle_summary_df)
shuffle_eval_df.to_csv(os.path.join(OUTPUT_DIR, "shuffle_ablation_raw.csv"), index=False)
shuffle_summary_df.to_csv(os.path.join(OUTPUT_DIR, "shuffle_ablation_summary.csv"), index=False)

In [ ]:
# Figure 4 — shuffle ablation

for metric in ["coef_rmse", "field_rmse", "traj_rmse"]:
    pivot = shuffle_summary_df.pivot(index="base_model", columns="metadata_condition", values=metric)
    fig, ax = plt.subplots(figsize=(8, 4))
    pivot.plot(kind="bar", ax=ax)
    ax.set_ylabel(metric)
    ax.set_title(f"Shuffle ablation — {metric}")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"figure4_shuffle_ablation_{metric}.png"), dpi=220, bbox_inches="tight")
    plt.show()

ratio_rows = []
for base_model, sub in shuffle_summary_df.groupby("base_model"):
    true = sub[sub["metadata_condition"] == "true"].iloc[0]
    shuffled = sub[sub["metadata_condition"] == "shuffled"].iloc[0]

    for metric in ["coef_rmse", "field_rmse", "traj_rmse"]:
        ratio_rows.append({
            "base_model": base_model,
            "metric": metric,
            "true": true[metric],
            "shuffled": shuffled[metric],
            "shuffle_to_true_ratio": shuffled[metric] / max(true[metric], 1e-12),
        })

shuffle_ratio_df = pd.DataFrame(ratio_rows)
display(shuffle_ratio_df)
shuffle_ratio_df.to_csv(os.path.join(OUTPUT_DIR, "shuffle_to_true_ratio.csv"), index=False)

## 10. Final summary table

Paper-ready comparison:

| model | mean coef RMSE | mean field RMSE | mean traj RMSE | universality score | win rate |

In [ ]:
mean_metrics = holdout_summary_df.groupby("model")[["coef_rmse", "field_rmse", "traj_rmse"]].mean().reset_index()

final_summary = mean_metrics.merge(
    universality_score_df[["model", "universality_score", "win_rate"]],
    on="model",
    how="left",
)

final_summary = final_summary.sort_values(
    ["universality_score", "win_rate", "traj_rmse"],
    ascending=[False, False, True],
)

display(final_summary)
final_summary.to_csv(os.path.join(OUTPUT_DIR, "final_ml_baseline_summary.csv"), index=False)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ["coef_rmse", "field_rmse", "traj_rmse"]):
    ax.bar(final_summary["model"], final_summary[metric])
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure5_final_ml_summary.png"), dpi=220, bbox_inches="tight")
plt.show()

## 11. Paper-ready conclusion paragraph

In [ ]:
best_model = final_summary.iloc[0]["model"]
best_u = final_summary.iloc[0]["universality_score"]
best_win = final_summary.iloc[0]["win_rate"]

symbolic_row = final_summary[final_summary["model"] == "pruned_symbolic"].iloc[0]

conclusion = f'''
## ML baseline comparison

We compared the pruned symbolic chart against dense ridge regression and flexible ML baselines, including random forests, gradient boosting, and a small MLP. All models used the same metadata-derived feature inputs and were evaluated on hard holdout blocks. The strongest overall model by universality score was `{best_model}` with Uτ={best_u:.3f} and win rate={best_win:.3f}. The pruned symbolic chart achieved Uτ={symbolic_row["universality_score"]:.3f}, win rate={symbolic_row["win_rate"]:.3f}, mean trajectory RMSE={symbolic_row["traj_rmse"]:.4f}, and retained direct interpretability as a sparse coordinate chart. Flexible ML baselines may fit particular blocks, but the symbolic chart remains the primary interpretable model and provides a structured benchmark for regime-shift generalization.
'''

with open(os.path.join(OUTPUT_DIR, "ml_baseline_conclusion.md"), "w", encoding="utf-8") as f:
    f.write(conclusion.strip() + "\n")

display(Markdown(conclusion))

## 12. Export manifest

In [ ]:
manifest = {
    "data_source": data_source,
    "synthetic_fallback": bool(USE_SYNTHETIC_FALLBACK),
    "regime_count": int(len(coef_df)),
    "models": list(MODEL_FUNCS.keys()),
    "universality_tolerance": U_TOL,
    "selected_shuffle_models": selected_shuffle_models,
    "outputs": sorted(os.listdir(OUTPUT_DIR)),
}

with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

display(pd.DataFrame({"output_file": sorted(os.listdir(OUTPUT_DIR))}))
print("Saved outputs under:", OUTPUT_DIR)

## Final statement

Notebook 66 answers the main external-baseline question:

> Does the sparse symbolic chart remain competitive against flexible ML baselines under regime shift?

Use this notebook as the robustness supplement after Notebook 64–65.

Recommended next step:

**Notebook 67 — uncertainty intervals and bootstrap confidence for universality score**